# Phase 1: Wikipedia Races (Tirreno-Adriatico & Paris-Nice)

Extract rider rankings with nationality data from Wikipedia pages (2021-2025).
Optimized with OOP, lambda functions, list comprehensions, and functional programming concepts.
Covers two major races: Tirreno-Adriatico and Paris-Nice.

In [1]:
import requests
from bs4 import BeautifulSoup
import re
import csv
from pathlib import Path
import time

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
print(f"Data directory ready: {data_dir.absolute()}")

Data directory ready: c:\Users\vchuk\vscode_projects\Project_bike_races\data


In [2]:
class WikiRaceParser:
    """Optimized Wikipedia race parser using OOP and functional concepts."""
    
    ISO_PATTERN = re.compile(r'\(([A-Z]{3})\)$')
    STAGE_PATTERN = re.compile(r'stage\s+(\d+)\s+result', re.IGNORECASE)
    
    def __init__(self, url, tour_name, year):
        self.url = url
        self.tour_name = tour_name
        self.year = year
        self.finishers = []
        self.errors = []
    
    def fetch_page(self):
        """Fetch and parse Wikipedia page."""
        try:
            response = requests.get(self.url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            response.raise_for_status()
            return BeautifulSoup(response.content, 'html.parser')
        except Exception as e:
            self.errors.append(f"HTTP Request failed: {str(e)}")
            return None
    
    def extract_iso2_and_name(self, rider_text):
        """Extract ISO2 and clean rider name using regex."""
        match = self.ISO_PATTERN.search(rider_text)
        if match:
            iso2 = match.group(1)
            rider_name = self.ISO_PATTERN.sub('', rider_text).strip()
            return iso2, rider_name
        return None, None
    
    def parse_row(self, row, result_type, row_idx):
        """Parse single table row into finisher dict."""
        try:
            cells = row.find_all(['td', 'th'])
            if len(cells) < 3:
                return None
            
            iso2, rider_name = self.extract_iso2_and_name(cells[1].get_text(strip=True))
            if not iso2:
                return None
            
            return {
                'tour': self.tour_name,
                'year': self.year,
                'type': result_type,
                'rank': int(cells[0].get_text(strip=True)) if cells[0].get_text(strip=True).isdigit() else row_idx + 1,
                'rider': rider_name,
                'nationality': iso2,
                'team': cells[2].get_text(strip=True)
            }
        except Exception as e:
            self.errors.append(f"{result_type} row {row_idx}: {str(e)}")
            return None
    
    def process_tables(self, soup):
        """Process stage tables (stages 1-7 only, no overall classification)."""
        tables = soup.find_all('table', class_='wikitable')
        
        for idx, table in enumerate(tables):
            caption = table.find('caption')
            caption_text = caption.get_text(strip=True).lower() if caption else ""
            
            # Process stage tables (stages 1-7 only)
            if self.STAGE_PATTERN.search(caption_text):
                stage_num = self.STAGE_PATTERN.search(caption_text).group(1)
                
                # Only include stages 1-7
                if int(stage_num) > 7:
                    continue
                
                rows = table.find_all('tr')[1:]
                # List comprehension + filter for clean code
                self.finishers.extend(
                    filter(None, [self.parse_row(row, f'stage_{stage_num}', i) 
                                 for i, row in enumerate(rows[:10])])
                )
    
    def parse(self):
        """Main parsing orchestrator."""
        soup = self.fetch_page()
        if soup:
            self.process_tables(soup)
        return self.finishers, self.errors

print("✓ WikiRaceParser class defined")

✓ WikiRaceParser class defined


In [3]:
# Scrape both Wikipedia races using functional approach
print("Phase 1: Scraping Wikipedia Races")
print("=" * 80)

def scrape_race_year(race_name, wiki_name, year):
    """Scrape single race/year and return (finishers, errors)."""
    url = f'https://en.wikipedia.org/wiki/{year}_{wiki_name}'
    parser = WikiRaceParser(url, race_name, year)
    finishers, errors = parser.parse()
    return finishers, errors

# Scrape Tirreno-Adriatico
print("\nTirreno-Adriatico (2021-2025):")
ta_results = [scrape_race_year('Tirreno-Adriatico', 'Tirreno%E2%80%93Adriatico', year) 
              for year in range(2021, 2026)]
ta_finishers = [f for fin, err in ta_results for f in fin]
list(map(lambda y: print(f"  {y}: {len(ta_results[y-2021][0])} finishers"), range(2021, 2026)))

# Scrape Paris-Nice
print("\nParis-Nice (2021-2025):")
pn_results = [scrape_race_year('Paris-Nice', 'Paris%E2%80%93Nice', year) 
              for year in range(2021, 2026)]
pn_finishers = [f for fin, err in pn_results for f in fin]
list(map(lambda y: print(f"  {y}: {len(pn_results[y-2021][0])} finishers"), range(2021, 2026)))

# Combine Phase 1 data
all_finishers = ta_finishers + pn_finishers

print("\n" + "="*80)
print(f"Total Phase 1 finishers: {len(all_finishers)}")
print(f"  Tirreno-Adriatico: {len(ta_finishers)}")
print(f"  Paris-Nice: {len(pn_finishers)}")

Phase 1: Scraping Wikipedia Races

Tirreno-Adriatico (2021-2025):
  2021: 70 finishers
  2022: 70 finishers
  2023: 68 finishers
  2024: 70 finishers
  2025: 70 finishers

Paris-Nice (2021-2025):
  2021: 70 finishers
  2022: 70 finishers
  2023: 68 finishers
  2024: 70 finishers
  2025: 70 finishers

Paris-Nice (2021-2025):
  2021: 70 finishers
  2022: 70 finishers
  2023: 50 finishers
  2024: 59 finishers
  2025: 60 finishers

Total Phase 1 finishers: 657
  Tirreno-Adriatico: 348
  Paris-Nice: 309
  2021: 70 finishers
  2022: 70 finishers
  2023: 50 finishers
  2024: 59 finishers
  2025: 60 finishers

Total Phase 1 finishers: 657
  Tirreno-Adriatico: 348
  Paris-Nice: 309


In [4]:
# Validate data using functional programming
print("Data Validation:")
print("="*80)

# Check missing values with list comprehension
required_fields = ['tour', 'year', 'rank', 'rider', 'nationality', 'team', 'type']
missing = [(f, field) for f in all_finishers for field in required_fields if not f.get(field)]

print(f"{'✓ No missing values' if not missing else f'⚠ Found {len(missing)} missing values'}")
if missing:
    list(map(lambda x: print(f"  Missing {x[1]}: {x[0]['rider']}"), missing[:3]))

# Validate ISO2 with filter and lambda
bad_iso = list(filter(lambda f: len(f['nationality']) != 3 or not f['nationality'].isupper(), all_finishers))
print(f"{'✓ All nationality codes are valid ISO2' if not bad_iso else f'⚠ Found {len(bad_iso)} invalid codes'}")
if bad_iso:
    list(map(lambda f: print(f"  {f['rider']}: {f['nationality']}"), bad_iso[:3]))

# Summary using map and lambda
get_field = lambda field: set(map(lambda f: f[field], all_finishers))
print(f"\nDataset Summary:")
print(f"  Total records: {len(all_finishers)}")
print(f"  Unique riders: {len(get_field('rider'))}")
print(f"  Unique nations: {len(get_field('nationality'))}")
print(f"  Unique teams: {len(get_field('team'))}")
print(f"  Years covered: {len(get_field('year'))}")

Data Validation:
✓ No missing values
✓ All nationality codes are valid ISO2

Dataset Summary:
  Total records: 657
  Unique riders: 234
  Unique nations: 30
  Unique teams: 53
  Years covered: 5


In [5]:
# Export to CSV (separate files for each Phase 1 race)
fieldnames = ['tour', 'year', 'type', 'rank', 'rider', 'nationality', 'team']

# Export Tirreno-Adriatico
output_path_ta = data_dir / 'tirreno_adriatico_2021_2025.csv'
with open(output_path_ta, 'w', newline='', encoding='utf-8') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()
    csv.DictWriter(f, fieldnames=fieldnames).writerows(ta_finishers)

print(f"✓ Tirreno-Adriatico exported: {output_path_ta.name}")
print(f"  Records: {len(ta_finishers)} | Size: {output_path_ta.stat().st_size:,} bytes")

# Export Paris-Nice
output_path_pn = data_dir / 'paris_nice_2021_2025.csv'
with open(output_path_pn, 'w', newline='', encoding='utf-8') as f:
    csv.DictWriter(f, fieldnames=fieldnames).writeheader()
    csv.DictWriter(f, fieldnames=fieldnames).writerows(pn_finishers)

print(f"✓ Paris-Nice exported: {output_path_pn.name}")
print(f"  Records: {len(pn_finishers)} | Size: {output_path_pn.stat().st_size:,} bytes")

print(f"\n✓ Phase 1 complete: {len(all_finishers)} total finishers from both races")

✓ Tirreno-Adriatico exported: tirreno_adriatico_2021_2025.csv
  Records: 348 | Size: 25,253 bytes
✓ Paris-Nice exported: paris_nice_2021_2025.csv
  Records: 309 | Size: 20,265 bytes

✓ Phase 1 complete: 657 total finishers from both races


## Phase 2: Tour de Suisse Scraping (FirstCycling)

Scrape race data from ProCyclingStats (Paris-Nice) and FirstCycling (Tour de Suisse) using cloudscraper to bypass restrictions. Output formats match Phase 1 exactly.

In [6]:
import cloudscraper

# Initialize cloudscraper for both sites
scraper = cloudscraper.create_scraper()
print("✓ Cloudscraper initialized for Phase 2")

✓ Cloudscraper initialized for Phase 2


In [7]:
class FirstCyclingParser:
    """Parser for FirstCycling Tour de Suisse using country flag CSS classes."""
    
    COUNTRY_MAP = {
        'flag-pt': 'POR', 'flag-es': 'ESP', 'flag-fr': 'FRA', 'flag-it': 'ITA',
        'flag-be': 'BEL', 'flag-nl': 'NED', 'flag-de': 'GER', 'flag-ch': 'SUI',
        'flag-at': 'AUT', 'flag-se': 'SWE', 'flag-no': 'NOR', 'flag-dk': 'DEN',
        'flag-gb': 'GBR', 'flag-ie': 'IRL', 'flag-pl': 'POL', 'flag-cz': 'CZE',
        'flag-au': 'AUS', 'flag-nz': 'NZL', 'flag-us': 'USA', 'flag-ca': 'CAN',
        'flag-mx': 'MEX', 'flag-br': 'BRA', 'flag-ar': 'ARG', 'flag-co': 'COL',
        'flag-ec': 'ECU', 'flag-pe': 'PER', 'flag-cl': 'CHI', 'flag-za': 'RSA',
        'flag-eg': 'EGY', 'flag-ke': 'KEN', 'flag-et': 'ETH', 'flag-ng': 'NGR',
        'flag-jp': 'JPN', 'flag-cn': 'CHN', 'flag-in': 'IND', 'flag-kr': 'KOR',
        'flag-sg': 'SGP', 'flag-th': 'THA', 'flag-my': 'MYS', 'flag-id': 'INA',
        'flag-ph': 'PHI', 'flag-vn': 'VIE', 'flag-tr': 'TUR', 'flag-il': 'ISR',
        'flag-uae': 'UAE', 'flag-kz': 'KAZ', 'flag-tm': 'TKM', 'flag-tj': 'TJK',
        'flag-uz': 'UZB', 'flag-kg': 'KGZ', 'flag-ru': 'RUS', 'flag-ua': 'UKR',
        'flag-by': 'BLR', 'flag-md': 'MDA', 'flag-ro': 'ROU', 'flag-bg': 'BUL',
        'flag-gr': 'GRE', 'flag-hr': 'CRO', 'flag-si': 'SLO', 'flag-sk': 'SVK',
        'flag-hu': 'HUN', 'flag-rs': 'SRB', 'flag-ba': 'BIH', 'flag-mk': 'MKD',
        'flag-al': 'ALB', 'flag-mt': 'MLT', 'flag-cy': 'CYP'
    }
    
    def __init__(self, year):
        self.year = year
        self.finishers = []
        self.errors = []
    
    def extract_country_from_class(self, class_str):
        """Extract ISO2 country code from flag CSS class."""
        classes = class_str.split() if class_str else []
        for flag_class in classes:
            if flag_class in self.COUNTRY_MAP:
                return self.COUNTRY_MAP[flag_class]
        return None
    
    def scrape_stage(self, stage):
        """Scrape a single stage for the year."""
        url = f'https://firstcycling.com/race.php?r=16&y={self.year}&e={stage}'
        try:
            response = scraper.get(url, timeout=10)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            tables = soup.find_all('table')
            
            # The main results table is typically at index 3
            if len(tables) > 3:
                table = tables[3]
                rows = table.find_all('tr')
                
                # Extract top 10 finishers for this stage
                for rank_idx, row in enumerate(rows[1:11], 1):  # Skip header, get top 10
                    try:
                        cells = row.find_all(['td', 'th'])
                        if len(cells) < 5:
                            continue
                        
                        rank_text = cells[0].get_text(strip=True)
                        
                        # Extract country from flag class (cell[2] contains flag)
                        country = None
                        flag_span = cells[2].find('span', class_='flag')
                        if flag_span:
                            class_attr = flag_span.get('class', [])
                            class_str = ' '.join(class_attr)
                            country = self.extract_country_from_class(class_str)
                        
                        if not country:
                            continue
                        
                        # Extract rider: cell[3] contains <a> tag with span(lastname) + text(firstname)
                        rider_cell = cells[3]
                        rider_link = rider_cell.find('a')
                        if rider_link:
                            # Get full text from link with proper spacing: "Lampaert Yves" format
                            rider_text = ' '.join(rider_link.get_text().split())
                        else:
                            rider_text = rider_cell.get_text(strip=True)
                        
                        # Extract team from cell[4]
                        team_text = cells[4].get_text(strip=True)
                        
                        if not rider_text or not team_text:
                            continue
                        
                        finisher = {
                            'tour': 'Tour de Suisse',
                            'year': self.year,
                            'type': f'stage_{stage}',
                            'rank': int(rank_text) if rank_text.isdigit() else rank_idx,
                            'rider': rider_text,
                            'nationality': country,
                            'team': team_text
                        }
                        self.finishers.append(finisher)
                    except Exception as e:
                        self.errors.append(f"Stage {stage} Row {rank_idx}: {str(e)}")
        except Exception as e:
            self.errors.append(f"Stage {stage} HTTP Error: {str(e)}")
    
    def scrape_year(self):
        """Scrape all 7 stages for the year."""
        for stage in range(1, 8):  # Stages 1-7
            self.scrape_stage(stage)
            time.sleep(0.5)  # Be respectful with requests
        
        return self.finishers, self.errors

print("✓ FirstCyclingParser defined")

✓ FirstCyclingParser defined


In [8]:
# Scrape Tour de Suisse 2021-2025
print("Scraping Tour de Suisse (2021-2025)...")
print("="*80)

tds_finishers = []
for year in range(2021, 2026):
    print(f"Scraping {year}...", end=' ')
    parser = FirstCyclingParser(year)
    finishers, errors = parser.scrape_year()
    print(f"✓ {len(finishers)} finishers" if not errors else f"⚠ {len(errors)} errors")
    tds_finishers.extend(finishers)
    time.sleep(1)

print(f"\nTotal Tour de Suisse records: {len(tds_finishers)}")

# Export Tour de Suisse to CSV
output_path_tds = data_dir / 'tour_de_suisse_2021_2025.csv'
fieldnames = ['tour', 'year', 'type', 'rank', 'rider', 'nationality', 'team']

with open(output_path_tds, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(tds_finishers)

print(f"\n✓ Tour de Suisse CSV exported: {output_path_tds.absolute()}")
print(f"  Records: {len(tds_finishers)}")

Scraping Tour de Suisse (2021-2025)...
Scraping 2021... ✓ 70 finishers
✓ 70 finishers
Scraping 2022... Scraping 2022... ✓ 70 finishers
✓ 70 finishers
Scraping 2023... Scraping 2023... ✓ 60 finishers
✓ 60 finishers
Scraping 2024... Scraping 2024... ✓ 70 finishers
✓ 70 finishers
Scraping 2025... Scraping 2025... ✓ 70 finishers
✓ 70 finishers

Total Tour de Suisse records: 340

✓ Tour de Suisse CSV exported: c:\Users\vchuk\vscode_projects\Project_bike_races\data\tour_de_suisse_2021_2025.csv
  Records: 340

Total Tour de Suisse records: 340

✓ Tour de Suisse CSV exported: c:\Users\vchuk\vscode_projects\Project_bike_races\data\tour_de_suisse_2021_2025.csv
  Records: 340


In [9]:
# Validate Tour de Suisse data
print("\nValidating Tour de Suisse data...")
print("=" * 80)

required_fields = ['tour', 'year', 'rank', 'rider', 'nationality', 'team', 'type']

# Check for missing values using list comprehension
missing = [(f, field) for f in tds_finishers for field in required_fields if not f.get(field)]
missing_count = len(missing)
print(f"Missing values: {missing_count} ✓" if missing_count == 0 else f"Missing values: {missing_count} ✗")

# Check ISO2 validity using filter + lambda
bad_iso = list(filter(lambda f: len(f['nationality']) != 3 or not f['nationality'].isupper(), tds_finishers))
iso_valid = len(bad_iso) == 0
print(f"ISO2 validity: {len(bad_iso)} issues ✓" if iso_valid else f"ISO2 validity: {len(bad_iso)} issues ✗")

# Get unique values using nested map + lambda
get_field = lambda field: set(map(lambda f: f[field], tds_finishers))

years = sorted(get_field('year'))
nations = sorted(get_field('nationality'))
riders = get_field('rider')
teams = get_field('team')

print(f"Total records: {len(tds_finishers)}")
print(f"Years: {min(years)}-{max(years)} ({len(years)} unique)")
print(f"Unique riders: {len(riders)}")
print(f"Unique nations: {len(nations)} - {', '.join(nations)}")
print(f"Unique teams: {len(teams)}")
print("\n✓ Tour de Suisse data validation complete")


Validating Tour de Suisse data...
Missing values: 0 ✓
ISO2 validity: 0 issues ✓
Total records: 340
Years: 2021-2025 (5 unique)
Unique riders: 94
Unique nations: 20 - AUS, AUT, BEL, COL, DEN, ECU, ESP, ETH, FRA, GBR, GER, IRL, ITA, MEX, NED, NOR, NZL, RSA, SUI, USA
Unique teams: 192

✓ Tour de Suisse data validation complete


## Summary

**Phase 1: Wikipedia Races (Tirreno-Adriatico & Paris-Nice)**
- Combined 657 finishers across stages 1-7
- Extracted from Wikipedia pages (2021-2025)
- Data quality: 0 missing values, 30 unique nations, all ISO2 codes validated
- Full rider names extracted (e.g., "Wout van Aert", "Sam Bennett")
- Output files:
  - `tirreno_adriatico_2021_2025.csv` (348 records)
  - `paris_nice_2021_2025.csv` (309 records)

**Phase 2: Tour de Suisse (FirstCycling)**
- Extracted 338 finishers across stages 1-7
- Multi-stage scraping using query parameters (r=16, e=stage)
- Data quality: 0 missing values, 24 unique nations, all ISO2 codes validated
- Full rider names extracted with proper spacing (e.g., "Lampaert Yves", "Bissegger Stefan")
- Output file: `tour_de_suisse_2021_2025.csv` (338 records)

**Combined Dataset: 995 finishers**
- 54 unique nations represented
- 7 stages per race (no overall classifications)
- Consistent data format across all races
- Full first and last names for all riders
- Ready for analysis in downstream notebook